In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import pandas as pd

import projects.minigrid_repro.analysis_utils as a_utils

experiment_dir = r"C:\Users\77019\Downloads\gradient-routing-main\gradient-routing-main\projects\minigrid_repro\data\oversight_levels"
experiment_dir = r"C:\Users\77019\Downloads\selected_folders_holdouts\data\oversight_levels"
experiment_dir = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\1perc_all"
experiment_dir = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\1perc_paper"
print("Reading files...", end=" ")
eval_files = glob.glob(os.path.join(experiment_dir, "eval_results*.csv"))
eval_dfs = [(pd.read_csv(file), file) for file in eval_files]

In [ ]:
    # if df["run_label"].iat[0] == "naive_outcomes+earlystop_0.002" and df['oversight_prob'].iat[0] == 0.008:
    # if df['run_label'][0] == "routing" and df['oversight_prob'][0] == 0.01:


In [ ]:
def get_avg_return(df):
    final_steps = (
        df[df.update_idx <= 20000]
        .sort_values("update_idx")
        .groupby(["run_label", "oversight_prob", "run_id"])
        .tail(1)
    )
    res = (
        final_steps.groupby(["run_label", "oversight_prob"])
        .agg({"avg_return": ["mean", a_utils.ci_width]})
        .reset_index()
    )
    return abs(res['avg_return']['mean'][0])

In [ ]:
names = []
for df, name in eval_dfs:
    if df['run_label'][0] == "routing+baseline" and df['oversight_prob'][0] == 0.01:
        # if get_avg_return(df) >= 0.2:
        # if df['update_idx'].max() < 10000:
            names.append(name.split('\\')[-1])

names.sort()

In [ ]:
names

In [ ]:
import os
import shutil   # <-- for file copying

# ── 1. Decide where you want the copies ─────────────────────────────
dest_dir = r"C:\Users\77019\Downloads\oversight_selected"
os.makedirs(dest_dir, exist_ok=True)          # makes it if it isn’t there

# ── 2. Copy each file we filtered for ───────────────────────────────
for df, src_path in eval_dfs:
    if df["run_label"].iat[0] == "naive_outcomes+earlystop_0.002" and df['oversight_prob'].iat[0] == 0.008:
        shutil.copy2(src_path,                    # source file (full path)
                     os.path.join(dest_dir,
                                  os.path.basename(src_path)))  # keep original name
        print(f"Copied {os.path.basename(src_path)} → {dest_dir}")


In [ ]:
import pandas as pd
profs_df = pd.read_csv(r"C:\Users\77019\Downloads\AI Safety Professionals.csv")

In [ ]:
type(profs_df[profs_df['Taking students?'].notna()]['Taking students?'][10])

In [ ]:
my_profs_df = profs_df[  (profs_df['Taking students?'] == 'checked') 
                       & (profs_df['Open Positions'])] 
my_profs_df = profs_df[profs_df['Taking students?'] == 'checked'] 

In [ ]:
my_profs_df

проверить графики холдаута на рандом ранах

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import projects.minigrid_repro.analysis_utils as a_utils

import glob
import pandas as pd

experiment_dir = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\rnd_hps"
experiment_dir = r"C:\Users\77019\Downloads\selected_folders_rnd_14\data\oversight_levels"
description = experiment_name = "rnd_hp"
smooth_amt = 1

import os
import pandas as pd
import glob
import matplotlib.pyplot as plt

# Load all train_results
train_files = glob.glob(os.path.join(experiment_dir, "train_results*.csv"))
train_res = pd.concat([pd.read_csv(f) for f in train_files])

# Load all eval_results
eval_files = glob.glob(os.path.join(experiment_dir, "eval_results*.csv"))
eval_res = pd.concat([pd.read_csv(f) for f in eval_files])

holdout_files = glob.glob(os.path.join(experiment_dir, "holdout_results*.csv"))
holdout_res = pd.concat([pd.read_csv(f) for f in holdout_files])

metrics_oversight = {}
for level in eval_res.oversight_prob.unique():
    metrics_oversight[level] = []

# Get all unique run_ids
run_ids = sorted(train_res.run_id.unique())

# Figure layout: rows = n_runs, cols = 3 (train, holdout, test)
n_runs = len(run_ids)
figsize = (12, 4 * n_runs)
fig, axes = plt.subplots(nrows=n_runs, ncols=3, figsize=figsize)
fig.suptitle(f"{description} ({n_runs} total runs)")

# metrics_oversight = 

# Loop through runs
for i, run_id in enumerate(run_ids):
    # TRAIN
    ax_train = axes[i, 0] if n_runs > 1 else axes[0]
    subset_train = train_res[train_res.run_id == run_id]
    a_utils.gplot(
        subset_train,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_train,
    )
    ax_train.set_title(f"Run {run_id} - Train")
    ax_train.set_xlabel("Update step")
    ax_train.set_ylabel("Train Return")

    # HOLDOUT
    subset_holdout = holdout_res[holdout_res.run_id == run_id]
    ax_holdout = axes[i, 1] if n_runs > 1 else axes[1]
    a_utils.gplot(
        subset_holdout,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_holdout,
    )
    ax_holdout.set_title(f"Run {run_id} - Holdout")
    ax_holdout.set_xlabel("Update step")
    ax_holdout.set_ylabel("Holdout Return")

    # EVAL
    ax_eval = axes[i, 2] if n_runs > 1 else axes[2]
    subset_eval = eval_res[eval_res.run_id == run_id]
    a_utils.gplot(
        subset_eval,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_eval,
    )
    ax_eval.set_title(f"Run {run_id} - Eval")
    ax_eval.set_xlabel("Update step")
    ax_eval.set_ylabel("Eval Return")

    best_holdout_update_idx = subset_holdout.loc[subset_holdout["avg_return"].idxmax()]["update_idx"]
    ax_holdout.axvline(best_holdout_update_idx, color='red', linestyle='--', label='Best Update')

    metrics_oversight[subset_eval.oversight_prob.iloc[0]].append(subset_eval[subset_eval.update_idx == best_holdout_update_idx].avg_return.values[0])
# plt.tight_layout()
# plt.savefig(
#     os.path.join(
#         figures_dir,
#         f"rl_per_run_curves_{training_method}_{oversight_percent}.pdf",
#     ),
#     bbox_inches="tight",
# )



In [ ]:
for ovs_porb, metrics in metrics_oversight.items():
    print(ovs_porb, sum(metrics)/len(metrics))

In [ ]:
import glob, os
import pandas as pd

# 1) Load eval & holdout
eval_res    = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(experiment_dir,"eval_results*.csv"))])
holdout_res = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(experiment_dir,"holdout_results*.csv"))], ignore_index=True)

# 2) Merge in each run’s best‐holdout update
best_idx = (
    holdout_res
      .loc[holdout_res.groupby("run_id")["avg_return"].idxmax(),
           ["run_id","update_idx"]]
      .rename(columns={"update_idx":"best_update"})
)
eval_res = (
    eval_res
      .merge(best_idx, on="run_id", how="left")
      .query("best_update.isna() or update_idx <= best_update")
      .drop(columns=["best_update"])
)

# 3) Final‐step selection (one row per run)
final_steps = (
    eval_res
      .sort_values("update_idx")
      .groupby(["run_label","oversight_prob","run_id"], as_index=False)
      .tail(1)
)

# 4) Compute the oversight‐level means
means = final_steps.groupby("oversight_prob")["avg_return"].mean()
print(means)


In [ ]:
best_idx = pd.DataFrame(holdout_res.groupby("run_id")["avg_return"].idxmax()).rename(columns={"update_idx": "best_update"})

In [ ]:
holdout_res

In [ ]:
best_indexes = holdout_res.groupby("run_id")["avg_return"].idxmax()
best_rows = holdout_res.loc[best_indexes, ["run_id", "update_idx"]]
best_rows = best_rows.reset_index(drop=True)


In [ ]:
best_indexes

making holdouts from old verson

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import projects.minigrid_repro.analysis_utils as a_utils

import glob
import pandas as pd

experiment_dir = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\rnd_hps"
description = experiment_name = "rnd_hp"
smooth_amt = 1

import os
import pandas as pd
import glob
import matplotlib.pyplot as plt

# Load all train_results
train_files = glob.glob(os.path.join(experiment_dir, "train_results*.csv"))
train_res = pd.concat([pd.read_csv(f) for f in train_files])

# Load all eval_results
eval_files = glob.glob(os.path.join(experiment_dir, "eval_results*.csv"))
eval_res = pd.concat([pd.read_csv(f) for f in eval_files])

In [ ]:
# holdout_files = glob.glob(os.path.join(experiment_dir, "holdout_results*.csv"))
# holdout_res = pd.concat([pd.read_csv(f) for f in holdout_files])

# metrics_oversight = {}
# for level in eval_res.oversight_prob.unique():
#     metrics_oversight[level] = []

# Get all unique run_ids
run_ids = sorted(train_res.run_id.unique())

# Figure layout: rows = n_runs, cols = 3 (train, holdout, test)
n_runs = len(run_ids)
figsize = (12, 4 * n_runs)
fig, axes = plt.subplots(nrows=n_runs, ncols=3, figsize=figsize)
fig.suptitle(f"{description} ({n_runs} total runs)")

# metrics_oversight = 

# Loop through runs
for i, run_id in enumerate(run_ids):
    # TRAIN
    ax_train = axes[i, 0] if n_runs > 1 else axes[0]
    subset_train = train_res[train_res.run_id == run_id]
    a_utils.gplot(
        subset_train,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_train,
    )
    ax_train.set_title(f"Run {run_id} - Train")
    ax_train.set_xlabel("Update step")
    ax_train.set_ylabel("Train Return")

    # HOLDOUT
    subset_holdout = holdout_res[holdout_res.run_id == run_id]
    ax_holdout = axes[i, 1] if n_runs > 1 else axes[1]
    a_utils.gplot(
        subset_holdout,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_holdout,
    )
    ax_holdout.set_title(f"Run {run_id} - Holdout")
    ax_holdout.set_xlabel("Update step")
    ax_holdout.set_ylabel("Holdout Return")

    # EVAL
    ax_eval = axes[i, 2] if n_runs > 1 else axes[2]
    subset_eval = eval_res[eval_res.run_id == run_id]
    a_utils.gplot(
        subset_eval,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_eval,
    )
    ax_eval.set_title(f"Run {run_id} - Eval")
    ax_eval.set_xlabel("Update step")
    ax_eval.set_ylabel("Eval Return")

    best_holdout_update_idx = subset_holdout.loc[subset_holdout["avg_return"].idxmax()]["update_idx"]
    ax_holdout.axvline(best_holdout_update_idx, color='red', linestyle='--', label='Best Update')

    metrics_oversight[subset_eval.oversight_prob.iloc[0]].append(subset_eval[subset_eval.update_idx == best_holdout_update_idx].avg_return.values[0])
# plt.tight_layout()
# plt.savefig(
#     os.path.join(
#         figures_dir,
#         f"rl_per_run_curves_{training_method}_{oversight_percent}.pdf",
#     ),
#     bbox_inches="tight",
# )



multiple runs auto work 

In [15]:
import os
import glob
import pandas as pd
import shutil

# --- Setup ---
input_dir = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\rnd_paper"
output_base_dir = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\rnd_paper_split"
os.makedirs(output_base_dir, exist_ok=True)

folder = r'C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\rnd_paper_split'
if folder == input_dir:
    raise KeyError("no")

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # remove file or symlink
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # remove folder and its contents
    except Exception as e:
        print(f"❌ Failed to delete {file_path}: {e}")

eval_files = glob.glob(os.path.join(input_dir, "eval_results_*.csv"))

def parse_oversight_folder(file_path):
    try:
        df = pd.read_csv(file_path)
        row = df.iloc[0]
        oversight_prob = row["oversight_prob"]
        run_label = row["run_label"]
        oversight_holdout = float(run_label.split("_")[-1])
        combined = oversight_prob + oversight_holdout
        return round(combined, 4)
    except Exception as e:
        print(f"⚠️ Skipping {file_path}: {e}")
        return None

# --- Main loop ---
for eval_file in eval_files:
    run_id = os.path.basename(eval_file).replace("eval_results_", "").replace(".csv", "")
    oversight_val = parse_oversight_folder(eval_file)
    if oversight_val is None:
        continue

    folder = os.path.join(output_base_dir, str(oversight_val))
    os.makedirs(folder, exist_ok=True)

    # Copy eval
    shutil.copy2(eval_file, os.path.join(folder, os.path.basename(eval_file)))

    # Copy train
    train_file = os.path.join(input_dir, f"train_results_{run_id}.csv")
    if os.path.exists(train_file):
        shutil.copy2(train_file, os.path.join(folder, os.path.basename(train_file)))

    # Copy holdout
    holdout_file = os.path.join(input_dir, f"holdout_results_{run_id}.csv")
    if os.path.exists(holdout_file):
        shutil.copy2(holdout_file, os.path.join(folder, os.path.basename(holdout_file)))

print("✅ Done. Grouped by oversight and all related files copied.")


✅ Done. Grouped by oversight and all related files copied.


In [16]:
import os

parent_path = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\rnd_paper_split"
subfolders = [name for name in os.listdir(parent_path)
              if os.path.isdir(os.path.join(parent_path, name))]

print(subfolders)



['0.001', '0.003', '0.01', '0.025', '0.05', '0.1', '0.2', '0.4', '0.8']


In [13]:
import os
import shutil

python_path = r'C:/Users/77019/pyver/py312/python.exe'

folder = r'C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\figures\rnd_paper_split'
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # remove file or symlink
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # remove folder and its contents
    except Exception as e:
        print(f"❌ Failed to delete {file_path}: {e}")

for oversight_level in subfolders:
# for oversight_level in ['0.025']:
    oversight_folder = f'rnd_paper_split/{oversight_level}'
    
    !{python_path} -m projects.minigrid_repro.analyze_multiple_runs --exp_name {oversight_folder} --subset_to_oversight none --show_legend

Reading files... subsetting training points... done.


c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x000001A17DBDBD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function std at 0x000001A17DBDBEC0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x000001A17DBDBD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be use

Reading files... subsetting training points... done.


c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x00000186F05FBD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function std at 0x00000186F05FBEC0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x00000186F05FBD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be use

Reading files... 


...

c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x00000230DA85BD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function std at 0x00000230DA85BEC0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x00000230DA85BD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be use


Reading files... subsetting training points... done.


c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x000001F2B2F1BD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function std at 0x000001F2B2F1BEC0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x000001F2B2F1BD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be use

Reading files... subsetting training points... done.


c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x00000247BE65BD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function std at 0x00000247BE65BEC0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x00000247BE65BD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be use

by oversight and holdout

In [28]:
import os
import glob
import pandas as pd
import shutil

oversight_folder = 0.001

# --- Setup ---
input_dir = fr"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\rnd_paper_split\{oversight_folder}"

eval_files = glob.glob(os.path.join(input_dir, "eval_results_*.csv"))
eval_dfs = [pd.read_csv(file) for file in eval_files]
eval_res = pd.concat(eval_dfs)

oversight_levels = eval_res['oversight_prob'].unique()
oversight_levels

array([0.0005, 0.0008, 0.0007])

In [29]:
import os
import shutil

python_path = r'C:/Users/77019/pyver/py312/python.exe'
oversight_folder = f'rnd_paper_split/{oversight_folder}'


for ovs_level in oversight_levels:
    !{python_path} -m projects.minigrid_repro.analyze_multiple_runs --exp_name {oversight_folder} --subset_to_oversight {ovs_level} --show_legend

Reading files... subsetting training points... done.


c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x00000293FF10BD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function std at 0x00000293FF10BEC0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  .agg(


Reading files... subsetting training points... done.


c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x000002AC1BC8BD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function std at 0x000002AC1BC8BEC0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  .agg(


Reading files... subsetting training points... done.


c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function mean at 0x000001CE7EFCBD80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(
c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analysis_utils.py:126: FutureWarning: The provided callable <function std at 0x000001CE7EFCBEC0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  .agg(
